# S4_2 — Masking Strategy Design and Visualisation
Demonstrates both masking approaches:
1. **Standard multi-block masking** — I-JEPA original (ablation baseline)
2. **Disease-region-biased masking** — novel contribution (hue-saliency weighted)

Produces dissertation-ready figures for Chapter 3 (Methodology).


In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import torch
import matplotlib.pyplot as plt

from stage4_leaf_jepa_pretraining.config_stage4 import *
from stage4_leaf_jepa_pretraining.pretrain_utils import (
    set_seed, PlantVillagePretrainDataset, get_pretrain_transform,
    MultiBlockMasking, DiseaseRegionBiasedMasking, SaliencyMap
)

set_seed(RANDOM_SEED)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("S4_2 Masking Strategy Visualisation")


In [ ]:
# ── Cell 2: Load sample images ────────────────────────────────────────────────
transform = get_pretrain_transform(NORM_MEAN, NORM_STD, IMAGE_CROP, IMAGE_RESIZE)
csv_path  = SPLITS_DIR / "plantvillage_splits.csv"
dataset   = PlantVillagePretrainDataset(csv_path, transform=transform)

# Sample diverse images (diseased and healthy)
sample_indices = [i for i in range(0, min(len(dataset), 5000), 500)][:8]
sample_imgs    = torch.stack([dataset[i][0] for i in sample_indices])
print(f"Sample images loaded: {sample_imgs.shape}")


In [ ]:
# ── Cell 3: Initialise masking strategies ─────────────────────────────────────
# Standard masking (I-JEPA original)
standard_masking = MultiBlockMasking(
    image_size        = IMAGE_CROP,
    patch_size        = PATCH_SIZE,
    num_target_blocks = PT_NUM_TARGET_BLOCKS,
    context_scale     = PT_CONTEXT_SCALE,
    context_ratio     = PT_CONTEXT_RATIO,
    target_scale      = PT_TARGET_SCALE,
    target_ratio      = PT_TARGET_RATIO,
)

# Disease-region-biased masking (novel contribution)
biased_masking = DiseaseRegionBiasedMasking(
    image_size        = IMAGE_CROP,
    patch_size        = PATCH_SIZE,
    num_target_blocks = PT_NUM_TARGET_BLOCKS,
    context_scale     = PT_CONTEXT_SCALE,
    context_ratio     = PT_CONTEXT_RATIO,
    target_scale      = PT_TARGET_SCALE,
    target_ratio      = PT_TARGET_RATIO,
    bias_strength     = SALIENCY_BIAS_STRENGTH,
)

# Saliency function
saliency_fn = SaliencyMap(
    patch_size         = PATCH_SIZE,
    image_size         = IMAGE_CROP,
    healthy_hue_center = HEALTHY_HUE_CENTER,
    healthy_hue_sigma  = HEALTHY_HUE_SIGMA,
)

print("✅ Masking strategies initialised")
print(f"  Standard: MultiBlockMasking, {PT_NUM_TARGET_BLOCKS} targets")
print(f"  Biased:   DiseaseRegionBiasedMasking, bias_strength={SALIENCY_BIAS_STRENGTH}")


In [ ]:
# ── Cell 4: Visualise standard masking ────────────────────────────────────────
# Show 4 examples of standard masking
fig, axes_grid = plt.subplots(4, 3, figsize=(15, 20))
for row_i, img_t in enumerate(sample_imgs[:4]):
    ctx_mask, tgt_masks = standard_masking()
    # Inline simple visualisation
    H = W = IMAGE_CROP // PATCH_SIZE
    mean = torch.tensor(NORM_MEAN).view(3, 1, 1)
    std  = torch.tensor(NORM_STD).view(3, 1, 1)
    img_vis = (img_t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    overlay_ctx = img_vis.copy()
    for i in range(H * W):
        if not ctx_mask[i]:
            r, c = i // W, i % W
            overlay_ctx[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] *= 0.3

    import matplotlib.cm as cm
    colours = cm.Set1(np.linspace(0, 0.8, len(tgt_masks)))
    overlay_tgt = img_vis.copy() * 0.4
    for tm, col in zip(tgt_masks, colours):
        for i in range(H * W):
            if tm[i]:
                r, c = i // W, i % W
                overlay_tgt[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] =                     overlay_tgt[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] * 0.1 +                     np.array(col[:3]) * 0.9

    axes_grid[row_i, 0].imshow(img_vis)
    axes_grid[row_i, 0].set_title("Original" if row_i == 0 else "")
    axes_grid[row_i, 0].axis("off")
    axes_grid[row_i, 1].imshow(overlay_ctx)
    axes_grid[row_i, 1].set_title(f"Context ({ctx_mask.sum().item()} patches)" if row_i == 0 else "")
    axes_grid[row_i, 1].axis("off")
    axes_grid[row_i, 2].imshow(overlay_tgt)
    axes_grid[row_i, 2].set_title(f"Targets ({len(tgt_masks)} blocks)" if row_i == 0 else "")
    axes_grid[row_i, 2].axis("off")

fig.suptitle("Standard I-JEPA Multi-Block Masking (Ablation Baseline)", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "S4_standard_masking_examples.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✅ Standard masking figure saved")


In [ ]:
# ── Cell 5: Visualise saliency maps ───────────────────────────────────────────
fig, axes = plt.subplots(4, 3, figsize=(15, 20))
for row_i, img_t in enumerate(sample_imgs[:4]):
    saliency = saliency_fn(img_t, NORM_MEAN, NORM_STD)
    H = W = IMAGE_CROP // PATCH_SIZE
    mean = torch.tensor(NORM_MEAN).view(3, 1, 1)
    std  = torch.tensor(NORM_STD).view(3, 1, 1)
    img_vis = (img_t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    sal_map = saliency.reshape(H, W)
    sal_up  = np.repeat(np.repeat(sal_map, PATCH_SIZE, 0), PATCH_SIZE, 1)
    sal_norm= (sal_up - sal_up.min()) / (sal_up.max() - sal_up.min() + 1e-8)

    axes[row_i, 0].imshow(img_vis)
    axes[row_i, 0].axis("off")
    axes[row_i, 0].set_title("Image" if row_i == 0 else "")

    im = axes[row_i, 1].imshow(sal_map, cmap="hot", interpolation="nearest")
    axes[row_i, 1].axis("off")
    axes[row_i, 1].set_title("Hue-Deviation Saliency" if row_i == 0 else "")

    axes[row_i, 2].imshow(img_vis)
    axes[row_i, 2].imshow(sal_norm, cmap="Reds", alpha=0.45)
    axes[row_i, 2].axis("off")
    axes[row_i, 2].set_title("Overlay" if row_i == 0 else "")

fig.suptitle("Disease-Region Saliency Map\n(Hue deviation from healthy leaf distribution — Novel Contribution)", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "S4_saliency_examples.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✅ Saliency map figure saved → use in Chapter 3 (Methodology)")


In [ ]:
# ── Cell 6: Visualise biased masking ──────────────────────────────────────────
fig, axes = plt.subplots(4, 3, figsize=(15, 20))
for row_i, img_t in enumerate(sample_imgs[:4]):
    saliency = saliency_fn(img_t, NORM_MEAN, NORM_STD)
    ctx_mask, tgt_masks = biased_masking(saliency=saliency)

    H = W = IMAGE_CROP // PATCH_SIZE
    mean = torch.tensor(NORM_MEAN).view(3, 1, 1)
    std  = torch.tensor(NORM_STD).view(3, 1, 1)
    img_vis = (img_t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    overlay_ctx = img_vis.copy()
    for i in range(H * W):
        if not ctx_mask[i]:
            r, c = i // W, i % W
            overlay_ctx[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] *= 0.3

    import matplotlib.cm as cm
    colours = cm.Set1(np.linspace(0, 0.8, len(tgt_masks)))
    overlay_tgt = img_vis.copy() * 0.4
    for tm, col in zip(tgt_masks, colours):
        for i in range(H * W):
            if tm[i]:
                r, c = i // W, i % W
                overlay_tgt[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] =                     overlay_tgt[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] * 0.1 +                     np.array(col[:3]) * 0.9

    axes[row_i, 0].imshow(img_vis)
    axes[row_i, 0].axis("off")
    axes[row_i, 0].set_title("Image" if row_i == 0 else "")
    axes[row_i, 1].imshow(overlay_ctx)
    axes[row_i, 1].axis("off")
    axes[row_i, 1].set_title("Context (biased)" if row_i == 0 else "")
    axes[row_i, 2].imshow(overlay_tgt)
    axes[row_i, 2].axis("off")
    axes[row_i, 2].set_title("Targets (disease-biased)" if row_i == 0 else "")

fig.suptitle("Disease-Region-Biased I-JEPA Masking (Novel Contribution)\nTargets preferentially placed over disease lesion regions", fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "S4_biased_masking_examples.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("✅ Biased masking figure saved")
print("\nKey comparison for dissertation (Figure 3.x):")
print("  Left panel:  Standard masking → targets randomly distributed")
print("  Right panel: Biased masking   → targets cluster over lesion regions")


In [ ]:
# ── Cell 7: Quantify masking coverage statistics ──────────────────────────────
import numpy as np

print("Computing masking coverage statistics over 500 random samples...")
set_seed(RANDOM_SEED)

std_coverage, biased_coverage = [], []
std_target_counts, biased_target_counts = [], []

for i in range(500):
    img_t = sample_imgs[i % len(sample_imgs)]
    saliency = saliency_fn(img_t, NORM_MEAN, NORM_STD)

    ctx_std, tgt_std = standard_masking()
    ctx_bia, tgt_bia = biased_masking(saliency=saliency)

    total = ctx_std.shape[0]
    std_coverage.append(ctx_std.sum().item() / total)
    biased_coverage.append(ctx_bia.sum().item() / total)

    # Average saliency of target patches vs all patches
    tgt_std_sal = np.mean([saliency[tm.nonzero(as_tuple=False).squeeze(-1).numpy()].mean()
                            for tm in tgt_std if tm.sum() > 0])
    tgt_bia_sal = np.mean([saliency[tm.nonzero(as_tuple=False).squeeze(-1).numpy()].mean()
                            for tm in tgt_bia if tm.sum() > 0])
    std_target_counts.append(tgt_std_sal)
    biased_target_counts.append(tgt_bia_sal)

print(f"\nContext coverage (standard):  {np.mean(std_coverage)*100:.1f}% ± {np.std(std_coverage)*100:.1f}%")
print(f"Context coverage (biased):    {np.mean(biased_coverage)*100:.1f}% ± {np.std(biased_coverage)*100:.1f}%")
print(f"\nTarget avg saliency (standard): {np.mean(std_target_counts):.4f}")
print(f"Target avg saliency (biased):   {np.mean(biased_target_counts):.4f}")
ratio = np.mean(biased_target_counts) / (np.mean(std_target_counts) + 1e-8)
print(f"  Saliency ratio biased/standard: {ratio:.2f}x")
print(f"\n  → Biased masking selects {ratio:.1f}x more salient (disease-like) patches as targets")
print("  → This quantifies the contribution of the novel masking strategy")
